In [1]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


!pip install -q kaggle

In [2]:
!kaggle datasets download -d kaustubhdikshit/neu-surface-defect-database
!unzip -q neu-surface-defect-database.zip

Dataset URL: https://www.kaggle.com/datasets/kaustubhdikshit/neu-surface-defect-database
License(s): unknown
100% 26.4M/26.4M [00:03<00:00, 8.77MB/s]



In [3]:
import os, time, copy, random
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from sklearn.metrics import classification_report, confusion_matrix



**Hyperparameters**

In [12]:
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TRAIN_DIR = "/content/NEU-DET/train/images"
VAL_DIR   = "/content/NEU-DET/validation/images"

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 5
LR = 1e-4
NUM_CLASSES = 6

#MEAN = np.array([0.485, 0.456, 0.406])
#STD  = np.array([0.229, 0.224, 0.225])
#These were computed from the ImageNet dataset.
#ImageNet has ~1.2 million images
#Researchers calculated the average pixel value (mean) and spread (std) for each RGB channel
#These became the standard normalization values used across deep learning



**Augmentations**

In [13]:
class NEUDataset(Dataset):
    def __init__(self, root, augment=False):
        self.paths = []
        self.labels = []
        self.class_names = sorted(os.listdir(root))
        for i, c in enumerate(self.class_names):
            for f in os.listdir(os.path.join(root, c)):
                self.paths.append(os.path.join(root, c, f))
                self.labels.append(i)
        self.augment = augment

    def __len__(self):
        return len(self.paths)

    def augment_img(self, img):
        if random.random() < 0.5:
            img = cv2.flip(img, 1)
        if random.random() < 0.5:
            img = cv2.flip(img, 0)
        if random.random() < 0.5:
            angle = random.uniform(-15, 15)
            h, w = img.shape[:2]
            M = cv2.getRotationMatrix2D((w//2, h//2), angle, 1)
            img = cv2.warpAffine(img, M, (w, h))
        if random.random() < 0.5:
            alpha = 1 + random.uniform(-0.3, 0.3)
            beta = random.uniform(-20, 20)
            img = cv2.convertScaleAbs(img, alpha=alpha, beta=beta)
        return img

    def __getitem__(self, idx):
        img = cv2.imread(self.paths[idx], cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)

        if self.augment:
            img = self.augment_img(img)

        img = img / 255.0
       # img = (img - MEAN) / STD
        img = np.transpose(img, (2, 0, 1))

        return torch.tensor(img, dtype=torch.float32), self.labels[idx]



**Architectures**

In [14]:
train_ds = NEUDataset(TRAIN_DIR, augment=True)
val_ds   = NEUDataset(VAL_DIR, augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

CLASS_NAMES = train_ds.class_names

def build_resnet50():
    m = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    m.fc = nn.Sequential(nn.Dropout(0.4), nn.Linear(m.fc.in_features, NUM_CLASSES))
    return m

def build_googlenet():
    m = models.googlenet(weights=models.GoogLeNet_Weights.DEFAULT, aux_logits=True)
    m.fc = nn.Sequential(nn.Dropout(0.4), nn.Linear(m.fc.in_features, NUM_CLASSES))
    m.aux1.fc2 = nn.Linear(1024, NUM_CLASSES)
    m.aux2.fc2 = nn.Linear(1024, NUM_CLASSES)
    return m

def build_efficientnet():
    m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
    in_f = m.classifier[1].in_features
    m.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(in_f, NUM_CLASSES))
    return m

MODELS = {
    "ResNet50": build_resnet50,
    "GoogLeNet": build_googlenet,
    "EfficientNet": build_efficientnet
}

criterion = nn.CrossEntropyLoss()



**Training**

In [15]:
def train_epoch(model, loader, optimizer, is_gnet=False):
    model.train()
    loss_sum = correct = total = 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        out = model(imgs)

        if is_gnet and hasattr(out, "logits"):
            loss = (criterion(out.logits, labels) +
                    0.3 * criterion(out.aux_logits2, labels) +
                    0.3 * criterion(out.aux_logits1, labels))
            preds = out.logits.argmax(1)
        else:
            loss = criterion(out, labels)
            preds = out.argmax(1)

        loss.backward()
        optimizer.step()

        loss_sum += loss.item() * imgs.size(0)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return loss_sum / total, correct / total

@torch.no_grad()
def eval_epoch(model, loader):
    model.eval()
    loss_sum = correct = total = 0
    all_p, all_l = [], []

    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        out = model(imgs)
        loss = criterion(out, labels)
        preds = out.argmax(1)

        loss_sum += loss.item() * imgs.size(0)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        all_p.extend(preds.cpu().numpy())
        all_l.extend(labels.cpu().numpy())

    return loss_sum / total, correct / total, all_p, all_l

def train_model(name, build_fn):
    model = build_fn().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LR)
    best_acc = 0
    best_w = None
    is_gnet = (name == "GoogLeNet")

    for ep in range(EPOCHS):
        tl, ta = train_epoch(model, train_loader, optimizer, is_gnet)
        vl, va, _, _ = eval_epoch(model, val_loader)

        if va > best_acc:
            best_acc = va
            best_w = copy.deepcopy(model.state_dict())

        print(f"{name} | Epoch {ep+1} | Train Acc {ta:.4f} | Val Acc {va:.4f}")

    model.load_state_dict(best_w)
    return model, best_acc



**Results**

In [16]:
results = {}

for name, fn in MODELS.items():
    model, acc = train_model(name, fn)
    _, _, preds, labels = eval_epoch(model, val_loader)

    print(f"\n{name}")
    print(classification_report(labels, preds, target_names=CLASS_NAMES))

    results[name] = acc

print("\nFinal Results:")
for k, v in results.items():
    print(f"{k}: {v*100:.2f}%")

ResNet50 | Epoch 1 | Train Acc 0.7917 | Val Acc 0.7806
ResNet50 | Epoch 2 | Train Acc 0.9819 | Val Acc 0.9750
ResNet50 | Epoch 3 | Train Acc 0.9924 | Val Acc 0.9972
ResNet50 | Epoch 4 | Train Acc 0.9896 | Val Acc 0.9889
ResNet50 | Epoch 5 | Train Acc 0.9965 | Val Acc 0.9917

ResNet50
                 precision    recall  f1-score   support

        crazing       1.00      1.00      1.00        60
      inclusion       1.00      0.98      0.99        60
        patches       1.00      1.00      1.00        60
 pitted_surface       1.00      1.00      1.00        60
rolled-in_scale       1.00      1.00      1.00        60
      scratches       0.98      1.00      0.99        60

       accuracy                           1.00       360
      macro avg       1.00      1.00      1.00       360
   weighted avg       1.00      1.00      1.00       360



/usr/local/lib/python3.12/dist-packages/torchvision/models/googlenet.py:341: UserWarning: auxiliary heads in the pretrained googlenet model are NOT pretrained, so make sure to train them
  warnings.warn(


GoogLeNet | Epoch 1 | Train Acc 0.7993 | Val Acc 0.9861
GoogLeNet | Epoch 2 | Train Acc 0.9826 | Val Acc 0.9944
GoogLeNet | Epoch 3 | Train Acc 0.9875 | Val Acc 0.9972
GoogLeNet | Epoch 4 | Train Acc 0.9931 | Val Acc 1.0000
GoogLeNet | Epoch 5 | Train Acc 0.9944 | Val Acc 1.0000

GoogLeNet
                 precision    recall  f1-score   support

        crazing       1.00      1.00      1.00        60
      inclusion       1.00      1.00      1.00        60
        patches       1.00      1.00      1.00        60
 pitted_surface       1.00      1.00      1.00        60
rolled-in_scale       1.00      1.00      1.00        60
      scratches       1.00      1.00      1.00        60

       accuracy                           1.00       360
      macro avg       1.00      1.00      1.00       360
   weighted avg       1.00      1.00      1.00       360

EfficientNet | Epoch 1 | Train Acc 0.8056 | Val Acc 0.9361
EfficientNet | Epoch 2 | Train Acc 0.9812 | Val Acc 0.9944
EfficientNet | Epo

**To check if there is a data leak**

In [17]:
train_files = set([os.path.basename(p) for p in train_ds.paths])
val_files   = set([os.path.basename(p) for p in val_ds.paths])

common = train_files.intersection(val_files)

print("Common files:", len(common))

Common files: 0
